# 03 — Classification Model

Classify crowd levels into **Low / Medium / High** buckets using RandomForestClassifier (production model).

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')

from apps.predict.ml.features import engineer_features, BUCKET_MAP, REVERSE_BUCKET_MAP
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/ahmedabad_metro_bookings.csv')
if 'bucket' not in df.columns:
    def bucket_crowd(v):
        if v <= 50: return 'Low'
        elif v <= 150: return 'Medium'
        return 'High'
    df['bucket'] = df['actual_crowd'].apply(bucket_crowd)

X = engineer_features(df)
y = df['bucket'].map(REVERSE_BUCKET_MAP)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Class distribution: {dict(y.value_counts())}')

In [ ]:
# Compare classification models
classifiers = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
}

for name, clf in classifiers.items():
    clf.fit(X_train_s, y_train)
    acc = clf.score(X_test_s, y_test)
    cv = cross_val_score(clf, X_train_s, y_train, cv=5).mean()
    print(f'{name:25s} | Test Acc={acc:.4f} | 5-fold CV={cv:.4f}')

In [ ]:
# Detailed report for production model (RandomForest)
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_s, y_train)
y_pred = rf.predict(X_test_s)

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High'], zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Low', 'Medium', 'High'], ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix — RandomForestClassifier')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (top 15)
feature_names = list(X.columns)
importances = rf.feature_importances_
top_idx = np.argsort(importances)[-15:][::-1]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh([feature_names[i] for i in top_idx][::-1], importances[top_idx][::-1], color='#6366f1')
ax.set_xlabel('Importance')
ax.set_title('Top 15 Feature Importances — RandomForest')
plt.tight_layout()
plt.show()

In [ ]:
# Hyperparameter tuning
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train_s, y_train)
print(f'Best params: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')
print(f'Test accuracy: {grid.score(X_test_s, y_test):.4f}')